# Phase 4: Enterprise Fraud Intelligence Framework

## Section 1: Project Philosophy
Traditional feature selection removes features based on a single statistical cut-off. We are systematically evaluating **fraud intelligence**. Every candidate feature must prove its worth across orthogonal dimensions, culminating in a robust **Fraud Intelligence Index (FII)** and a **Feature Consensus Score**.

## Pipeline Flow
```text
3573 Features
      │
      ▼
Remove Constant Features
      │
      ▼
Variance Analysis
      │
      ▼
Mutual Information
      │
      ▼
Permutation Importance
      │
      ▼
XGBoost Gain
      │
      ▼
SHAP
      │
      ▼
Stability Analysis
      │
      ▼
Production Intelligence
      │
      ▼
Fraud Intelligence Index & Consensus
      │
      ▼
Tier Assignment
      │
      ▼
Approved Features
```

In [1]:
from src.utils.bootstrap import bootstrap
bootstrap()

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import MinMaxScaler
import xgboost as xgb
import shap
import time
import hashlib
from datetime import datetime

warnings.filterwarnings("ignore")

reports_dir = Path("../reports/phase_04")
reports_dir.mkdir(parents=True, exist_ok=True)
data_engineered_dir = Path("../data/engineered")
data_selected_dir = Path("../data/selected")
data_selected_dir.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
start_time = time.time()


## Section 2: Load Datasets

In [2]:
df = pd.read_csv(data_engineered_dir / 'feature_engineered_dataset.csv')
target_col = 'Fraud' if 'Fraud' in df.columns else ('Is_Fraud' if 'Is_Fraud' in df.columns else ('F3924' if 'F3924' in df.columns else 'Target'))

# Generate dataset hash for versioning
dataset_hash = hashlib.md5(df.to_csv(index=False).encode()).hexdigest()[:8]
print(f"Loaded dataset shape: {df.shape}")
print(f"Dataset Hash: {dataset_hash}")

Loaded dataset shape: (9082, 3576)
Dataset Hash: d609b263


## Section 3: Dataset Integrity

In [3]:
obj_cols = df.select_dtypes(include=['object', 'datetime']).columns
if len(obj_cols) > 0:
    df.drop(columns=obj_cols, inplace=True)

integrity = {
    'Rows': len(df),
    'Columns': len(df.columns),
    'Target Exists': target_col in df.columns,
    'No Leakage': 'F3912' not in df.columns,
    'No Objects': df.select_dtypes(include=['object']).shape[1] == 0,
    'Duplicates': df.duplicated().sum(),
}
for k,v in integrity.items():
    assert (v is True or v == 0) if isinstance(v, bool) or k == 'Duplicates' else True
    print(f"{k}: {v}")


Rows: 9082
Columns: 3575
Target Exists: True
No Leakage: True
No Objects: True
Duplicates: 0


## Section 4: Variance & Missingness Robustness

In [4]:
variances = df.var()
zero_var = variances[variances == 0].index.tolist()
if zero_var:
    df.drop(columns=zero_var, inplace=True)
    variances.drop(index=zero_var, inplace=True)

missing_pct = df.isnull().mean()
missing_robustness = 1.0 - missing_pct

var_df = pd.DataFrame({'Feature': variances.index, 'Variance': variances.values, 'Missing_Pct': missing_pct[variances.index], 'Missing_Robustness': missing_robustness[variances.index]})


## Section 5: Correlation Penalty
Absolute Pearson Correlation logic:
- >0.95 -> Penalty 0.25
- 0.90-0.95 -> Penalty 0.10
- Otherwise -> 0

In [5]:
sample_df = df.sample(n=min(2000, len(df)), random_state=SEED)
X_sample = sample_df.drop(columns=[target_col])
corr_matrix = X_sample.corr().abs()

avg_corr = corr_matrix.mean()

def compute_penalty(corr_val):
    if corr_val > 0.95: return 0.25
    elif corr_val > 0.90: return 0.10
    else: return 0.0

# Apply penalty based on maximum correlation with any other feature (excluding self)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
max_corr = upper.max(axis=0).fillna(upper.max(axis=1)).fillna(0)

penalties = max_corr.apply(compute_penalty)
corr_score = 1.0 - penalties

corr_df = pd.DataFrame({'Feature': corr_matrix.columns, 'Correlation_Penalty': penalties, 'Correlation_Score': corr_score})


## Section 6: Mutual Information

In [6]:
X = df.drop(columns=[target_col])
y = df[target_col]
mi_sample = df.sample(n=min(3000, len(df)), random_state=SEED)
X_mi = mi_sample.drop(columns=[target_col])
y_mi = mi_sample[target_col]

mi_scores = mutual_info_classif(X_mi.fillna(-999), y_mi, random_state=SEED)
mi_df = pd.DataFrame({'Feature': X_mi.columns, 'MI_Score': mi_scores})


## Section 7: XGBoost Gain

In [7]:
model = xgb.XGBClassifier(n_estimators=100, max_depth=4, random_state=SEED, eval_metric='logloss')
model.fit(X, y)
gain = model.get_booster().get_score(importance_type='gain')

xgb_df = pd.DataFrame({
    'Feature': list(gain.keys()),
    'XGB_Gain': list(gain.values())
})


## Section 8: Permutation Importance

In [8]:
perm_sample_X = X.sample(n=min(1000, len(X)), random_state=SEED)
perm_sample_y = y.loc[perm_sample_X.index]
perm_results = permutation_importance(model, perm_sample_X, perm_sample_y, n_repeats=3, random_state=SEED, n_jobs=-1)

perm_df = pd.DataFrame({
    'Feature': perm_sample_X.columns,
    'Permutation_Score': perm_results.importances_mean
})


## Section 9: SHAP Explainability

In [9]:
shap_sample = X.sample(n=min(500, len(X)), random_state=SEED)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(shap_sample)

shap_mean_abs = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'Feature': shap_sample.columns, 'SHAP_Importance': shap_mean_abs})


## Section 10: Feature Stability
5-Fold Stratified Cross Validation -> Coefficient of Variation -> Normalized Stability Score

In [10]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
stability_scores = {feat: [] for feat in X.columns}

for train_idx, val_idx in skf.split(X, y):
    X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
    m = xgb.XGBClassifier(n_estimators=30, max_depth=3, random_state=SEED, eval_metric='logloss')
    m.fit(X_tr, y_tr)
    gains = m.get_booster().get_score(importance_type='gain')
    for feat in X.columns:
        stability_scores[feat].append(gains.get(feat, 0))

stab_df = pd.DataFrame({
    'Feature': list(stability_scores.keys()),
    'Stab_Mean': [np.mean(stability_scores[f]) for f in stability_scores.keys()],
    'Stab_Std': [np.std(stability_scores[f]) for f in stability_scores.keys()]
})
# Normalized Stability Score (1 / (1 + CoV))
stab_df['Stability_Score'] = np.where(stab_df['Stab_Mean'] > 0, 1 / (1 + (stab_df['Stab_Std'] / stab_df['Stab_Mean'])), 0)


## Section 11: Production Intelligence (Domain Rubric)
Business Interpretability (0-2), Operational Availability (0-2), Production Stability (0-2), Regulatory Explainability (0-2), Fraud Behaviour Alignment (0-2)

In [11]:
domain_scores = []
for f in X.columns:
    # Baseline for engineered features
    interp = 1.0
    avail = 2.0
    stab = 2.0
    explain = 1.0
    align = 1.0
    
    if 'Age' in f or 'Tenure' in f or 'Retail' in f or 'Missing' in f or 'Interaction' in f:
        interp = 2.0
        explain = 2.0
        align = 2.0
        
    total_score = (interp + avail + stab + explain + align) / 10.0
    
    business_cat = 'Unknown'
    if 'Age' in f or 'Tenure' in f: business_cat = 'Temporal'
    elif 'Missing' in f: business_cat = 'Missingness'
    elif 'Interaction' in f: business_cat = 'Engineered'
    elif 'Retail' in f or 'Corporate' in f: business_cat = 'Identity'
    
    domain_scores.append({'Feature': f, 'Domain_Score': total_score, 'Business_Category': business_cat})
biz_df = pd.DataFrame(domain_scores)


## Section 12: Consensus Score & Feature Confidence
Consensus: How many metrics rank the feature in their respective Top 20%?
Confidence: Agreement among all evaluators.

In [12]:
scaler = MinMaxScaler()

master = pd.DataFrame({'Feature': X.columns})
for df_ in [var_df, corr_df, mi_df, xgb_df, perm_df, shap_df, stab_df[['Feature', 'Stability_Score']], biz_df]:
    master = master.merge(df_, on='Feature', how='left').fillna(0)

scale_cols = ['MI_Score', 'Permutation_Score', 'XGB_Gain', 'SHAP_Importance', 
              'Stability_Score', 'Missing_Robustness', 'Correlation_Score', 'Domain_Score']
master[scale_cols] = scaler.fit_transform(master[scale_cols])

# Feature Consensus Score (4/4 metrics in top 20%)
q80_mi = master['MI_Score'].quantile(0.80)
q80_xgb = master['XGB_Gain'].quantile(0.80)
q80_shap = master['SHAP_Importance'].quantile(0.80)
q80_perm = master['Permutation_Score'].quantile(0.80)

master['Consensus_Score'] = (
    (master['MI_Score'] >= q80_mi).astype(int) +
    (master['XGB_Gain'] >= q80_xgb).astype(int) +
    (master['SHAP_Importance'] >= q80_shap).astype(int) +
    (master['Permutation_Score'] >= q80_perm).astype(int)
)
master['Consensus_Normalized'] = master['Consensus_Score'] / 4.0

# Sub-Scores
master['Statistical_Intelligence'] = (master['MI_Score'] + master['Permutation_Score']) / 2
master['Model_Intelligence'] = (master['XGB_Gain'] + master['SHAP_Importance']) / 2
master['Production_Intelligence'] = (master['Stability_Score'] + master['Missing_Robustness'] + master['Correlation_Score'] + master['Domain_Score']) / 4

# Enterprise FII
master['FII'] = (
    0.25 * master['Statistical_Intelligence'] +
    0.25 * master['Model_Intelligence'] +
    0.25 * master['Production_Intelligence'] +
    0.25 * master['Consensus_Normalized']
) * 100

# Feature Confidence Score (Agreement across all domains)
master['Confidence_Pct'] = (
    (master['Statistical_Intelligence'] > 0.5).astype(int) +
    (master['Model_Intelligence'] > 0.5).astype(int) +
    (master['Production_Intelligence'] > 0.5).astype(int) +
    (master['Consensus_Normalized'] > 0.5).astype(int)
) / 4.0 * 100

master.sort_values('FII', ascending=False, inplace=True)


## Section 13: Tier Assignment & Governance
Percentile thresholds: Top 1%, 5%, 15%, 35%

In [13]:
master['Percentile'] = master['FII'].rank(pct=True, ascending=False)

def assign_tier(pct):
    if pct <= 0.01: return 'Tier A (Mission Critical)'
    elif pct <= 0.05: return 'Tier B (Core)'
    elif pct <= 0.15: return 'Tier C (Strong)'
    elif pct <= 0.35: return 'Tier D (Useful)'
    else: return 'Tier E (Experimental)'

master['Tier'] = master['Percentile'].apply(assign_tier)

def assign_decision(tier):
    if 'Tier A' in tier or 'Tier B' in tier or 'Tier C' in tier: return 'Approved'
    elif 'Tier D' in tier: return 'Review'
    else: return 'Rejected'

def assign_reason(row):
    tier = row['Tier']
    if 'Tier A' in tier: return 'Irrefutable multi-dimensional support. Core detection component.'
    elif 'Tier B' in tier: return 'High predictive stability. Critical ensemble variable.'
    elif 'Tier C' in tier: return 'High statistical support, moderate model contribution, stable enough for ensemble learning.'
    elif 'Tier D' in tier: return 'Marginal intelligence, requires human oversight.'
    else: return 'Fails to meet minimum multivariate thresholds.'

master['Decision'] = master['Tier'].apply(assign_decision)
master['Reason'] = master.apply(assign_reason, axis=1)
master['Version'] = '1.0'
master['Timestamp'] = datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')

print(master['Tier'].value_counts())
print("\nDecisions:")
print(master['Decision'].value_counts())


Tier
Tier E (Experimental)        2408
Tier D (Useful)               630
Tier C (Strong)               357
Tier B (Core)                 143
Tier A (Mission Critical)      35
Name: count, dtype: int64

Decisions:
Decision
Rejected    2408
Review       630
Approved     535
Name: count, dtype: int64


## Section 14: Export & Machine-Readable Manifest (Versioned)

In [14]:
registry_cols = [
    'Feature', 'Missing_Pct', 'Variance', 'Business_Category', 
    'MI_Score', 'Permutation_Score', 'XGB_Gain', 'SHAP_Importance', 
    'Stability_Score', 'Correlation_Penalty', 'Consensus_Score', 
    'Confidence_Pct', 'FII', 'Tier', 'Decision', 'Reason', 
    'Version', 'Timestamp'
]
registry = master[registry_cols]
registry.to_csv(reports_dir / "fraud_intelligence_registry.csv", index=False)

approved_features = master[master['Decision'] == 'Approved']['Feature'].tolist()
final_df = df[approved_features + [target_col]]
final_df.to_csv(data_selected_dir / "approved_features.csv", index=False)

manifest = {
    "phase": "04",
    "version": "1.0",
    "run_date": datetime.utcnow().strftime('%Y-%m-%d'),
    "dataset_hash": dataset_hash,
    "notebook_version": "v5",
    "git_commit": "N/A",
    "execution_time_seconds": round(time.time() - start_time, 2),
    "input_dataset": "feature_engineered_dataset.csv",
    "output_dataset": "approved_features.csv",
    "total_input_features": len(X.columns),
    "total_approved_features": len(approved_features),
    "tier_distribution": master['Tier'].value_counts().to_dict(),
    "validation": "PASS"
}

with open(reports_dir / "evaluation_metadata.json", "w") as f:
    json.dump(manifest, f, indent=4)

print("Phase 4 completed. Manifest saved.")


Phase 4 completed. Manifest saved.
